In [ ]:
# --- ensure repo-root cwd so data/ paths resolve from anywhere ---
import os, sys
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src'))

# State of Health (SoH) for Mahindra vehicles — distance-per-SoC method

Mahindra telemetry has **no `batterySoh` field**, **no current channel**, and its `kwh` field
is a *signed instantaneous* power flow (+charging / −driving) that holds stale values when
parked — so it is **not integrable** into capacity. True coulomb counting (∫I·dt) and a clean
energy-count are both impossible here.

The clean, reliable signals are **`odometer`** (cumulative distance) and **`soc`**. So we estimate
SoH as **range retention**: during discharge (driving), usable capacity ∝ **km travelled per % SoC**
(`Δodometer / Δsoc`). As the pack fades, the vehicle covers fewer km per % SoC. We compute this per
month, normalise the earliest month to 100%, and apply a **monotonic non-increasing envelope**
(running minimum) — SoH never increases, as required.

Data comes from the monthly-sample extract in `data/mahindra_extracted/` (the 10 most-aged VINs).

## 1. Setup and load

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt

EXTRACTED = Path("data/mahindra/_archive/feed_oldest10")
SOH_OUT = Path("data/mahindra/soh"); SOH_OUT.mkdir(parents=True, exist_ok=True)

# Plausibility bounds for a single telemetry step during driving.
MIN_SOC, MAX_SOC = 1.0, 100.0
MAX_STEP_KM = 60.0      # max plausible km between two samples
MIN_SEG_SOC_DROP = 1.0  # ignore steps with <1% SoC change (noise)
KM_PER_SOC_BOUNDS = (0.2, 12.0)  # plausible km per 1% SoC for a Treo-class EV

cols = ["vin", "eventAt", "soc", "odometer", "kwh", "batteryTemp", "state"]
df = ds.dataset(EXTRACTED, format="parquet").to_table(columns=cols).to_pandas()
print(f"Loaded {len(df):,} rows, {df['vin'].nunique()} VINs")

df["eventAt"] = pd.to_datetime(df["eventAt"], unit="ms", errors="coerce")
for c in ["soc", "odometer", "kwh", "batteryTemp"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["vin", "eventAt", "soc", "odometer"])
df = df[(df["soc"] >= MIN_SOC) & (df["soc"] <= MAX_SOC) & (df["odometer"] > 0)]
print(f"After basic cleaning: {len(df):,} rows")

## 2. Per-step discharge metrics

For each VIN, sort by time and take consecutive-sample deltas. Keep **discharge steps**
(SoC falling, odometer advancing) that pass plausibility filters, and record `km_per_soc`
for each. These are the raw capacity observations.

In [ ]:
lo, hi = KM_PER_SOC_BOUNDS
parts = []
for vin, g in df.groupby("vin"):
    g = g.sort_values("eventAt")
    s = pd.DataFrame({
        "vin": vin,
        "eventAt": g["eventAt"].values,
        "d_odo": g["odometer"].diff().values,
        "soc_drop": (-g["soc"].diff()).values,   # positive when discharging
    })
    parts.append(s)

steps = pd.concat(parts, ignore_index=True).dropna(subset=["d_odo", "soc_drop"])
# Keep plausible discharge steps only.
steps = steps[(steps["soc_drop"] >= MIN_SEG_SOC_DROP) &
              (steps["d_odo"] > 0) & (steps["d_odo"] <= MAX_STEP_KM)]
steps["km_per_soc"] = steps["d_odo"] / steps["soc_drop"]
steps = steps[(steps["km_per_soc"] >= lo) & (steps["km_per_soc"] <= hi)]
steps["month"] = steps["eventAt"].dt.to_period("M").dt.to_timestamp()

print(f"{len(steps):,} valid discharge steps across {steps['vin'].nunique()} VINs")
print(steps["km_per_soc"].describe().round(2))

## 3. Monthly capacity → SoH (normalised + monotonic)

Per VIN per month, capacity proxy = **total discharge distance / total SoC consumed**
(a ratio of sums, robust to per-step noise). Normalise each VIN's earliest month to 100%,
then enforce a non-increasing curve with a running minimum.

In [ ]:
monthly = (steps.groupby(["vin", "month"])
                 .apply(lambda s: pd.Series({
                     "km_per_soc": s["d_odo"].sum() / s["soc_drop"].sum(),
                     "n_steps": len(s)}))
                 .reset_index())
# Require a minimum sample size per month-point to be trustworthy.
monthly = monthly[monthly["n_steps"] >= 5]

def to_soh(g):
    g = g.sort_values("month").copy()
    base = g["km_per_soc"].iloc[0]
    g["soh_raw"] = 100.0 * g["km_per_soc"] / base
    g["soh"] = g["soh_raw"].cummin()      # monotonic non-increasing envelope
    return g

soh = monthly.groupby("vin", group_keys=False).apply(to_soh)
print(f"SoH points: {len(soh)} across {soh['vin'].nunique()} VINs")
soh.head(12)

## 4. Plot SoH curves

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for vin, g in soh.groupby("vin"):
    ax.plot(g["month"], g["soh"], marker="o", ms=4, label=vin[-6:])
    ax.plot(g["month"], g["soh_raw"], ls=":", lw=0.8, alpha=0.4, color=ax.lines[-1].get_color())
ax.set_title("Mahindra SoH (range-retention, distance-per-SoC) — solid = monotonic, dotted = raw")
ax.set_ylabel("SoH (% of early-life km/%SoC)"); ax.set_xlabel("Month")
ax.grid(alpha=0.3); ax.legend(title="VIN tail", ncol=2, fontsize=8)
plt.tight_layout(); plt.show()

## 5. Per-VIN summary and save

In [ ]:
rows = []
for vin, g in soh.groupby("vin"):
    g = g.sort_values("month")
    months = (g["month"].iloc[-1] - g["month"].iloc[0]).days / 365.25
    drop = g["soh"].iloc[0] - g["soh"].iloc[-1]
    rows.append({
        "vin": vin,
        "months_span": round((g["month"].iloc[-1] - g["month"].iloc[0]).days / 30.4, 1),
        "soh_current_pct": round(float(g["soh"].iloc[-1]), 1),
        "soh_drop_pct": round(float(drop), 1),
        "degradation_pct_per_year": round(float(drop / months), 2) if months > 0 else np.nan,
        "n_points": len(g),
    })
summary = pd.DataFrame(rows).sort_values("soh_current_pct")
print(summary.to_string(index=False))

soh.to_parquet(SOH_OUT / "mahindra_soh_monthly.parquet", index=False)
summary.to_csv(SOH_OUT / "mahindra_soh_summary.csv", index=False)
print(f"\nSaved -> {SOH_OUT}/mahindra_soh_monthly.parquet and mahindra_soh_summary.csv")